# T2.1 – DBRepo Schema Creation (Final Version)
**Project:** Predicting the Market Value of Football Players  
**Owner:** Student D (ekene) — new database with full owner rights  
**Database ID:** `598ce585-d8b5-4a97-8f19-cb085d4a5b1e`

## Approach
Uses `CreateTableColumn` with explicit `ColumnType` values instead of
DataFrame dtype inference. This guarantees correct column types regardless
of how the SDK maps pandas dtypes internally.

## Column type mapping (schema.sql → ColumnType)
| schema.sql type | ColumnType used |
|---|---|
| `SMALLINT` | `ColumnType.SMALLINT` |
| `INT` | `ColumnType.INT` |
| `BIGINT` | `ColumnType.BIGINT` |
| `BOOLEAN` | `ColumnType.BOOL` |
| `DECIMAL(12,3)` | `ColumnType.DECIMAL` size=12 d=3 |
| `DECIMAL(16,2)` | `ColumnType.DECIMAL` size=16 d=2 |
| `DECIMAL(6,2)` | `ColumnType.DECIMAL` size=6 d=2 |
| `VARCHAR(n)` | `ColumnType.VARCHAR` size=n |
| `TEXT` | `ColumnType.TEXT` |

## Upload order
```
1. source_dataset  2. player  3. club  4. position  5. nationality  6. season
7. forward_player_valuation   8. transfer_value_observation
```

## 0. Install

In [1]:
# import sys
# !{sys.executable} -m pip install dbrepo==1.13.3 --quiet
# import dbrepo
# print(f"dbrepo: {dbrepo.__version__}")

## 1. Imports

In [2]:
from getpass import getpass
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import (
    CreateTableColumn,
    ColumnType,
    CreateTableConstraints,
)

## 2. Configuration

In [3]:
DBREPO_ENDPOINT  = "https://test.dbrepo.tuwien.ac.at"
USERNAME         = "ekene"
DATABASE_ID      = "598ce585-d8b5-4a97-8f19-cb085d4a5b1e"
IS_PUBLIC        = True
IS_SCHEMA_PUBLIC = True

## 3. Connect

In [4]:
password = getpass(f"DBRepo password for '{USERNAME}': ")
client = RestClient(
    endpoint=DBREPO_ENDPOINT,
    username=USERNAME,
    password=password,
)
print(f"Connected as: {client.whoami()}")

DBRepo password for 'ekene':  ········


ekene
Connected as: ekene


## 4. Define Column Specs

Each table is defined as a list of `CreateTableColumn` objects with explicit
`ColumnType` values — no DataFrame inference involved.

Helper shortcuts:
- `pk_smallint / pk_int / pk_bigint` — primary key columns
- `col_int / col_smallint` — non-nullable integer columns  
- `col_int_null / col_smallint_null` — nullable integer columns
- `col_bool / col_bool_null` — boolean columns
- `col_dec / col_dec_null` — decimal columns with size and precision
- `col_varchar / col_varchar_null` — varchar columns
- `col_text_null` — TEXT nullable column

In [5]:


def pk_smallint(name):
    return CreateTableColumn(name=name, type=ColumnType.SMALLINT, null_allowed=False)

def pk_int(name):
    return CreateTableColumn(name=name, type=ColumnType.INT, null_allowed=False)

def pk_bigint(name):
    return CreateTableColumn(name=name, type=ColumnType.BIGINT, null_allowed=False)

def col_smallint(name):
    return CreateTableColumn(name=name, type=ColumnType.SMALLINT, null_allowed=False)

def col_smallint_null(name):
    return CreateTableColumn(name=name, type=ColumnType.SMALLINT, null_allowed=True)

def col_int(name):
    return CreateTableColumn(name=name, type=ColumnType.INT, null_allowed=False)

def col_bool(name):
    return CreateTableColumn(name=name, type=ColumnType.BOOL, null_allowed=False)

def col_bool_null(name):
    return CreateTableColumn(name=name, type=ColumnType.BOOL, null_allowed=True)

def col_dec(name, size, d):
    return CreateTableColumn(name=name, type=ColumnType.DECIMAL,
                             null_allowed=False, size=size, d=d)

def col_dec_null(name, size, d):
    return CreateTableColumn(name=name, type=ColumnType.DECIMAL,
                             null_allowed=True, size=size, d=d)

def col_varchar(name, size=255):
    return CreateTableColumn(name=name, type=ColumnType.VARCHAR,
                             null_allowed=False, size=size)

def col_varchar_null(name, size=255):
    return CreateTableColumn(name=name, type=ColumnType.VARCHAR,
                             null_allowed=True, size=size)

def col_text_null(name):
    return CreateTableColumn(name=name, type=ColumnType.TEXT, null_allowed=True)

print("Helpers ready.")

Helpers ready.


In [6]:
# ── Table definitions ──────────────────────────────────────────────────────────
# Each matches schema.sql exactly.

TABLE_SPECS = {

    # ── source_dataset ────────────────────────────────────────────────────────
    # SMALLINT PK, 8x VARCHAR NOT NULL, 1x VARCHAR NULL, 1x TEXT NULL
    'source_dataset': [
        pk_smallint('source_dataset_id'),
        col_varchar('dataset_title',     255),
        col_varchar('original_creators', 255),
        col_varchar('publisher',         255),
        col_varchar_null('version_label', 50),
        col_varchar('doi',               100),
        col_varchar('doi_url',           255),
        col_varchar('license_name',      100),
        col_varchar('license_url',       255),
        col_varchar_null('source_filename', 255),
        col_text_null('notes'),
    ],

    # ── player ────────────────────────────────────────────────────────────────
    'player': [
        pk_int('player_id'),
        col_varchar('player_name', 255),
    ],

    # ── club ──────────────────────────────────────────────────────────────────
    'club': [
        pk_int('club_id'),
        col_varchar('club_name', 255),
    ],

    # ── position ──────────────────────────────────────────────────────────────
    'position': [
        pk_int('position_id'),
        col_varchar('position_name', 100),
    ],

    # ── nationality ───────────────────────────────────────────────────────────
    'nationality': [
        pk_int('nationality_id'),
        col_varchar('nationality_name', 100),
    ],

    # ── season ────────────────────────────────────────────────────────────────
    # SMALLINT PK (season year itself, not auto-increment)
    'season': [
        pk_smallint('season_year'),
        col_varchar_null('notes', 255),
    ],

    # ── forward_player_valuation ──────────────────────────────────────────────
    # Dataset 1: Forward football player valuation (Briseño & Rivera, 2024)
    # Target variable: market_value_mln
    'forward_player_valuation': [
        pk_int('forward_valuation_id'),
        col_smallint('source_dataset_id'),       # FK → source_dataset
        col_int('player_id'),                    # FK → player
        col_int('club_id'),                      # FK → club
        col_varchar('original_player_name', 255),
        col_varchar('original_team_name',   255),
        col_smallint('player_age_years'),
        col_dec('market_value_mln',      12, 3), # TARGET variable
        col_smallint('value_rank'),
        col_bool('plays_in_europe'),             # BOOLEAN
        col_int('matches_played'),
        col_int('goals'),
        col_int('assists'),
        col_int('minutes_per_goal'),
        col_int('minutes_played'),
        col_dec_null('instagram_followers_mln', 12, 3),  # 6 nulls in data
    ],

    # ── transfer_value_observation ────────────────────────────────────────────
    # Dataset 2: Transfer Value Determinants (Nisanov, 2025)
    # Target variable: value_end_mln
    # club_performance / relegation / success_or_not: NULL for 2020-2023 rows
    # start_value_eur: 17 NULLs, end_value_eur: 2 NULLs
    'transfer_value_observation': [
        pk_bigint('transfer_observation_id'),
        col_smallint('source_dataset_id'),           # FK → source_dataset
        col_int('player_id'),                        # FK → player
        col_int('position_id'),                      # FK → position
        col_int('nationality_id'),                   # FK → nationality
        col_int('club_id'),                          # FK → club
        col_smallint('season_year'),                 # FK → season
        col_varchar('original_player_name',      255),
        col_varchar('original_position_name',    100),
        col_varchar('original_nationality_name', 100),
        col_varchar('original_club_name',        255),
        col_smallint('age_then_years'),
        col_smallint('age_now_years'),
        col_smallint_null('club_performance'),       # NULL for 2020-2023
        col_bool_null('relegation'),                 # NULL for 2020-2023
        col_smallint_null('success_or_not'),         # NULL for 2020-2023
        col_int('total_games'),
        col_int('assists'),
        col_int('penalty_kicks'),
        col_int('total_minutes'),
        col_int('total_goals'),
        col_dec('height_cm',         6,  2),
        col_dec_null('start_value_eur', 16, 2),      # 17 NULLs in data
        col_dec_null('end_value_eur',   16, 2),      # 2 NULLs in data
        col_dec('delta_value_eur',   16,  2),
        col_dec('value_start_mln',   12,  3),
        col_dec('value_end_mln',     12,  3),        # TARGET variable
        col_dec('value_delta_mln',   12,  3),
    ],
}

print("Table specs defined:")
for name, cols in TABLE_SPECS.items():
    print(f"  {name}: {len(cols)} columns")

Table specs defined:
  source_dataset: 11 columns
  player: 2 columns
  club: 2 columns
  position: 2 columns
  nationality: 2 columns
  season: 2 columns
  forward_player_valuation: 16 columns
  transfer_value_observation: 28 columns


## 5. Table Descriptions

In [7]:
TABLE_DESCRIPTIONS = {
    'source_dataset': (
        'Provenance metadata for the two reused external source datasets. '
        'Records original creators, publisher (Mendeley Data), DOI, '
        'licence (CC BY 4.0), and source filename. '
        'The student group is not the original publisher or rights holder.'
    ),
    'player': (
        'Deduplicated lookup table of football player names '
        'appearing across both source datasets.'
    ),
    'club': (
        'Deduplicated lookup table of football club names '
        'appearing across both source datasets.'
    ),
    'position': (
        'Lookup table of football positions (e.g. Striker, Defensive Midfield) '
        'from the Transfer Value Determinants dataset.'
    ),
    'nationality': (
        'Lookup table of player nationality labels '
        'from the Transfer Value Determinants dataset.'
    ),
    'season': (
        'Season years (2019-2023) from the Transfer Value Determinants '
        'workbook sheets. Referenced by transfer_value_observation.'
    ),
    'forward_player_valuation': (
        'Normalised observations from the Forward Football Player Valuation '
        'dataset (Briseno and Rivera, 2024, DOI: 10.17632/cgc33scxg7.1). '
        'One row per forward player. Contains age, match statistics, '
        'Instagram followers, and market value in millions EUR. '
        'Target variable for ML: market_value_mln.'
    ),
    'transfer_value_observation': (
        'Normalised observations from the Transfer Value Determinants dataset '
        '(Nisanov, 2025, DOI: 10.17632/3btg6ptc7b.2). '
        'One row per player-season across 2019-2023. '
        'Contains performance stats, height, club context, and transfer values. '
        'Target variable for ML: value_end_mln. '
        'club_performance, relegation, success_or_not are NULL for 2020-2023.'
    ),
}

## 6. Create Tables
Creates tables in FK dependency order. Safe to re-run — skips existing tables.

In [8]:
# Primary key column name for each table
PRIMARY_KEYS = {
    'source_dataset':             'source_dataset_id',
    'player':                     'player_id',
    'club':                       'club_id',
    'position':                   'position_id',
    'nationality':                'nationality_id',
    'season':                     'season_year',
    'forward_player_valuation':   'forward_valuation_id',
    'transfer_value_observation': 'transfer_observation_id',
}

existing = {t.name: t.id for t in client.get_tables(database_id=DATABASE_ID)}
print(f"Already in database: {list(existing.keys()) or 'none'}")
print()

created_table_ids = dict(existing)

for table_name, columns in TABLE_SPECS.items():
    if table_name in existing:
        print(f"[SKIP] '{table_name}' (id: {existing[table_name]})")
        continue

    try:
        table = client.create_table(
            database_id=DATABASE_ID,
            name=table_name,
            columns=columns,
            constraints=CreateTableConstraints(
                primary_key=[PRIMARY_KEYS[table_name]],  # PK declared here
                uniques=[],
                checks=[],
                foreign_keys=[],
            ),
            description=TABLE_DESCRIPTIONS[table_name],
            is_public=IS_PUBLIC,
            is_schema_public=IS_SCHEMA_PUBLIC,
        )
        created_table_ids[table_name] = table.id
        print(f"[OK]   '{table_name}' created (id: {table.id})")

    except Exception as e:
        print(f"[FAIL] '{table_name}': {type(e).__name__}: {e}")

print()
print(f"Done. {len(created_table_ids)}/8 tables available.")

Already in database: ['nationality', 'position', 'club', 'player']

[FAIL] 'source_dataset': TypeError: RestClient.create_table() got an unexpected keyword argument 'columns'
[SKIP] 'player' (id: a4c1da46-c842-4fc2-bf68-7dcc45d79f80)
[SKIP] 'club' (id: 664015d2-9e5d-479d-8669-976799fe77f0)
[SKIP] 'position' (id: 3ef4099a-84a5-4b52-9e01-f064f664d3e1)
[SKIP] 'nationality' (id: 922db33d-5ac8-4473-859c-efe784e0c476)
[FAIL] 'season': TypeError: RestClient.create_table() got an unexpected keyword argument 'columns'
[FAIL] 'forward_player_valuation': TypeError: RestClient.create_table() got an unexpected keyword argument 'columns'
[FAIL] 'transfer_value_observation': TypeError: RestClient.create_table() got an unexpected keyword argument 'columns'

Done. 4/8 tables available.


## 7. Verify Column Types
Fetches each table and prints actual types registered in DBRepo.
Flags any column whose type doesn't match what schema.sql specifies.

In [10]:
# Expected types per column for the two critical fact tables
EXPECTED = {
    'forward_player_valuation': {
        'plays_in_europe':         'bool',
        'goals':                   'int',
        'assists':                 'int',
        'market_value_mln':        'decimal',
        'instagram_followers_mln': 'decimal',
        'player_age_years':        'smallint',
        'value_rank':              'smallint',
    },
    'transfer_value_observation': {
        'club_performance': 'smallint',
        'relegation':       'bool',
        'success_or_not':   'smallint',
        'height_cm':        'decimal',
        'start_value_eur':  'decimal',
        'end_value_eur':    'decimal',
        'value_end_mln':    'decimal',
    },
}

all_brief = client.get_tables(database_id=DATABASE_ID)
print(f"Total tables in database: {len(all_brief)}")
print("=" * 60)

for t_brief in all_brief:
    full = client.get_table(database_id=DATABASE_ID, table_id=t_brief.id)
    print(f"\n{full.name} ({len(full.columns)} cols):")
    for col in full.columns:
        flag = ""
        if full.name in EXPECTED and col.internal_name in EXPECTED[full.name]:
            exp = EXPECTED[full.name][col.internal_name]
            actual = str(col.type).lower()
            if exp not in actual:
                flag = f"  ← WRONG: expected {exp}, got {col.type}"
        print(f"  {col.internal_name}: {col.type}{flag}")

Total tables in database: 4

nationality (2 cols):
  nationality_id: ColumnType.BIGINT
  nationality_name: ColumnType.TEXT

position (2 cols):
  position_id: ColumnType.BIGINT
  position_name: ColumnType.TEXT

club (2 cols):
  club_id: ColumnType.BIGINT
  club_name: ColumnType.TEXT

player (2 cols):
  player_id: ColumnType.BIGINT
  player_name: ColumnType.TEXT


## 8. Print Table ID Map
Copy these into your upload notebook and views notebook.

In [11]:
all_brief = client.get_tables(database_id=DATABASE_ID)
table_id_map = {t.name: t.id for t in all_brief}

print(f'DATABASE_ID = "{DATABASE_ID}"')
print()
print("table_id_map = {")
for name, tid in table_id_map.items():
    print(f'    "{name}": "{tid}",')
print("}")

DATABASE_ID = "598ce585-d8b5-4a97-8f19-cb085d4a5b1e"

table_id_map = {
    "nationality": "922db33d-5ac8-4473-859c-efe784e0c476",
    "position": "3ef4099a-84a5-4b52-9e01-f064f664d3e1",
    "club": "664015d2-9e5d-479d-8669-976799fe77f0",
    "player": "a4c1da46-c842-4fc2-bf68-7dcc45d79f80",
}
